# Benchmark: Pandas vs Arrays/Listas vs `for` plano — búsqueda con `ddgs`

Dataset: MercadoLibre (`articuloTitulo`, `SKU`, `idPublicacion`, `zProductoLink`).

Por cada `articuloTitulo` se busca en DuckDuckGo (vía `ddgs`) un resumen y un posible link a PDF.

Se comparan 3 formas de recorrer el dataset:
1. **Pandas** → `Series.apply()` (sin `for`/`while` explícito)
2. **Arrays/listas** → `map()` + `functools.reduce()` (sin `for`/`while` explícito)
3. **For plano** → único bucle explícito del notebook

Y se mide el tiempo de ejecución al **10%**, **30%** y **50%** del dataset, con gráficos finales al 10% y al 50%.

In [1]:
!pip install -q ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 32.5 MB/s eta 0:00:00


## 1. Subir el CSV

Ejecutá esta celda y seleccioná el archivo `dataset_mercadolibre-scraper-espanol-castellano_...csv` desde tu computadora.
(Si preferís usar Google Drive en vez de subirlo a mano, reemplazá esta celda por un `drive.mount('/content/drive')` y ajustá `RUTA_CSV`.)

In [16]:
from google.colab import files

subida = files.upload()
RUTA_CSV = list(subida.keys())[0]
print("Archivo cargado:", RUTA_CSV)

Saving dataset_mercadolibre-scraper-espanol-castellano_2026-08-18_13-29-04-255.csv to dataset_mercadolibre-scraper-espanol-castellano_2026-08-18_13-29-04-255.csv
Archivo cargado: dataset_mercadolibre-scraper-espanol-castellano_2026-08-18_13-29-04-255.csv


## 2. Imports y configuración

In [29]:
import functools
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ddgs import DDGS

COLUMNAS_NECESARIAS = ["articuloTitulo", "SKU", "idPublicacion", "zProductoLink"]
BATCH_SIZE = 100            # tamaño de chunk para la carga batch
MAX_RESULTS_DDG = 3         # resultados a pedirle a ddgs por título
MAX_REINTENTOS = 2
DELAY_ENTRE_REQUESTS = 0.3  # segundos, para no ser bloqueados por rate limit

random.seed(42)

## 3. Carga batch

Se lee el CSV en *chunks* con `pandas` (no todo de una vez), pensado para cuando el dataset crezca de 478 filas a cientos de miles.

In [30]:
def cargar_dataset_batch(path_csv, batch_size=BATCH_SIZE):
    """Generador: entrega el CSV en chunks, ya recortado a las columnas necesarias."""
    for chunk in pd.read_csv(path_csv, chunksize=batch_size, encoding="utf-8-sig"):
        yield chunk[COLUMNAS_NECESARIAS]


def cargar_dataset_completo(path_csv, batch_size=BATCH_SIZE):
    """Concatena todos los batches en un único DataFrame."""
    batches = list(cargar_dataset_batch(path_csv, batch_size))
    return pd.concat(batches, ignore_index=True)


df = cargar_dataset_completo(RUTA_CSV)
print(f"Dataset cargado: {len(df)} filas")
df.head()

Dataset cargado: 478 filas


,articuloTitulo,SKU,idPublicacion,zProductoLink
0,Libro Fuego Y Sangre (canción De Hielo Y Fuego...,MLA22203654,MLA1438985527,https://www.mercadolibre.com.ar/libro-fuego-y-...
1,Mariana Enriquez Los peligros de fumar en la c...,MLA19864989,MLA1664274515,https://www.mercadolibre.com.ar/mariana-enriqu...
2,It Stephen King Español Debolsillo,MLA20913273,MLA2907767984,https://www.mercadolibre.com.ar/it-stephen-kin...
3,Socorro 12 Cuentos Para Caerse De Miedo: 12 cu...,MLA22090889,MLA2202982806,https://www.mercadolibre.com.ar/socorro-12-cue...
4,"El Exorcista, De Blatty, William Peter. Editor...",MLA26050668,MLA2610718784,https://www.mercadolibre.com.ar/el-exorcista-d...


## 4. Función atómica de búsqueda

Compartida por las 3 versiones. Nunca deja explotar una excepción: si falla una fila, no tira abajo el batch entero.

> El único `while` de todo el notebook está acá, y es solo el mecanismo de reintentos (acotado a `MAX_REINTENTOS`), no un bucle de iteración sobre el dataset.

In [43]:
import random

BACKENDS_FALLBACK = ["duckduckgo", "brave", "startpage", "mojeek", "yahoo"]

def buscar_resumen_ddg(titulo, max_results=MAX_RESULTS_DDG):
    """
    Busca información específica sobre `titulo` en DDGS.

    Devuelve (resumen, pdf_encontrado).
    """

    def _query_con_reintentos(consulta, etiqueta):
        """
        Ejecuta una query probando distintos backends en cascada.
        Si uno falla (timeout, rate limit, etc.), prueba el siguiente
        antes de esperar y reintentar desde el principio.
        """
        intento = 0
        while intento <= MAX_REINTENTOS:
            for backend in BACKENDS_FALLBACK:
                try:
                    resultados = list(
                        DDGS(timeout=20).text(
                            consulta,
                            max_results=max_results,
                            backend=backend,
                        )
                    )
                    if resultados:
                        return resultados
                except Exception as e:
                    print(f"[{etiqueta}] backend '{backend}' falló: "
                          f"{type(e).__name__}: {e}")
                    continue  # probar el siguiente backend sin esperar

            intento += 1
            espera = (2 ** intento) + random.uniform(0, 1)
            print(f"[{etiqueta}] ronda de intento {intento} agotó todos los "
                  f"backends — reintentando en {espera:.1f}s")
            time.sleep(espera)

        return []

    # ============================================================
    # 1. BÚSQUEDA ESPECÍFICA DE PDF
    # ============================================================
    consulta_pdf = f'"{titulo}" filetype:pdf'
    resultados_pdf = _query_con_reintentos(consulta_pdf, "pdf")

    pdf_encontrado = next(
        (
            resultado.get("href")
            for resultado in resultados_pdf
            if ".pdf" in resultado.get("href", "").lower()
        ),
        None
    )

    time.sleep(DELAY_ENTRE_REQUESTS + random.uniform(1, 3))

    # ============================================================
    # 2. BÚSQUEDA ESPECÍFICA DEL RESUMEN
    # ============================================================
    consulta_resumen = f'"{titulo}" libro resumen argumento sinopsis'
    resultados_resumen = _query_con_reintentos(consulta_resumen, "resumen")

    resumen = (
        resultados_resumen[0].get("body")
        if resultados_resumen
        else None
    )

    time.sleep(DELAY_ENTRE_REQUESTS + random.uniform(1, 3))

    return (resumen, pdf_encontrado)

## 5. Las 3 versiones

In [31]:
# --- 5a) VERSIÓN PANDAS -> sin for/while, usa apply() ---
def version_pandas(df):
    resultado = df.copy()
    pares = resultado["articuloTitulo"].apply(buscar_resumen_ddg)
    resultado["resumen"] = pares.apply(lambda p: p[0])
    resultado["pdf_encontrado"] = pares.apply(lambda p: p[1])
    return resultado

In [32]:
# --- 5b) VERSIÓN ARRAYS/LISTAS -> sin for/while, usa map() + reduce() ---
def version_arrays(df):
    titulos = np.array(df["articuloTitulo"])

    pares = list(map(buscar_resumen_ddg, titulos))

    resumenes = np.array(list(map(lambda p: p[0], pares)), dtype=object)
    pdfs = np.array(list(map(lambda p: p[1], pares)), dtype=object)

    columnas_extra = {"resumen": resumenes, "pdf_encontrado": pdfs}
    resultado = functools.reduce(
        lambda acc, kv: acc.assign(**{kv[0]: kv[1]}),
        columnas_extra.items(),
        df.copy(),
    )
    return resultado

In [33]:
# --- 5c) VERSIÓN FOR PLANO -> única versión con bucle explícito ---
def version_for(df):
    titulos = df["articuloTitulo"].tolist()
    resumenes = []
    pdfs = []

    for titulo in titulos:
        resumen, pdf = buscar_resumen_ddg(titulo)
        resumenes.append(resumen)
        pdfs.append(pdf)

    resultado = df.copy()
    resultado["resumen"] = resumenes
    resultado["pdf_encontrado"] = pdfs
    return resultado


VERSIONES = {
    "pandas": version_pandas,
    "arrays": version_arrays,
    "for_plano": version_for,
}

## 6. Muestreo por porcentaje

In [34]:
def muestrear(df, frac, seed=42):
    return df.sample(frac=frac, random_state=seed).reset_index(drop=True)

## 7. Guardado de datasets con metadata

Cada dataset enriquecido (con `resumen` y `pdf_encontrado`) se guarda en su propio CSV dentro de `datasets_generados/`, agregando columnas de metadata: versión usada, % de muestra, cantidad de filas, fecha/hora de la corrida, tiempo total y tiempo promedio por fila.

In [35]:
import os
from datetime import datetime

CARPETA_DATASETS = "datasets_generados"

def agregar_metadata(df_resultado, nombre_version, fraccion, segundos):
    n_filas = len(df_resultado)
    con_metadata = df_resultado.copy()
    con_metadata["meta_version"] = nombre_version
    con_metadata["meta_fraccion_dataset"] = fraccion
    con_metadata["meta_n_filas"] = n_filas
    con_metadata["meta_fecha_hora"] = datetime.now().isoformat(timespec="seconds")
    con_metadata["meta_segundos_totales"] = round(segundos, 4)
    con_metadata["meta_segundos_promedio_fila"] = round(segundos / n_filas, 4) if n_filas else None
    return con_metadata


def guardar_dataset(df_con_metadata, nombre_version, fraccion, carpeta=CARPETA_DATASETS):
    os.makedirs(carpeta, exist_ok=True)
    nombre_archivo = f"dataset_{nombre_version}_{int(fraccion*100)}pct.csv"
    ruta = os.path.join(carpeta, nombre_archivo)
    df_con_metadata.to_csv(ruta, index=False, encoding="utf-8-sig")
    return ruta

## 8. Benchmark

Corre las 3 versiones al 10%, 30% y 50% del dataset, mide el tiempo con `time.perf_counter()` y guarda cada dataset resultante con su metadata.

⚠️ Con dataset real esto hace requests de verdad a DuckDuckGo: al 50% (≈239 filas) puede tardar varios minutos y `ddgs` puede rate-limitear. Si eso pasa, bajá `MAX_RESULTS_DDG`, subí `DELAY_ENTRE_REQUESTS`, o probá primero solo con 10%.

In [39]:
def correr_benchmark(df, fracciones=(0.01, 0.02,0.03)):
    filas_resultado = []

    for frac in fracciones:
        muestra = muestrear(df, frac)
        for nombre_version, funcion in VERSIONES.items():
            inicio = time.perf_counter()
            df_resultado = funcion(muestra)
            elapsed = time.perf_counter() - inicio

            df_con_metadata = agregar_metadata(df_resultado, nombre_version, frac, elapsed)
            ruta_csv = guardar_dataset(df_con_metadata, nombre_version, frac)

            filas_resultado.append(
                {
                    "fraccion": frac,
                    "n_filas": len(muestra),
                    "version": nombre_version,
                    "segundos": elapsed,
                    "ruta_csv": ruta_csv,
                }
            )
            print(f"[{int(frac*100)}%  n={len(muestra):>3}] {nombre_version:<10} -> {elapsed:.2f}s -> {ruta_csv}")

    return pd.DataFrame(filas_resultado)

In [41]:
%rm -rf datasets_generados/

In [44]:

resultados = correr_benchmark(df, fracciones=(0.01, 0.02,0.03))
resultados.to_csv("resultados_benchmark.csv", index=False)
print("Datasets enriquecidos guardados en ./datasets_generados/")
resultados.pivot(index="version", columns="fraccion", values="segundos")

[pdf] backend 'duckduckgo' falló: DDGSException: No results found.
[pdf] backend 'brave' falló: DDGSException: No results found.
[pdf] backend 'startpage' falló: DDGSException: No results found.
[pdf] backend 'mojeek' falló: DDGSException: No results found.
[pdf] backend 'yahoo' falló: DDGSException: No results found.
[pdf] ronda de intento 1 agotó todos los backends — reintentando en 2.1s
[pdf] backend 'duckduckgo' falló: DDGSException: No results found.
[pdf] backend 'brave' falló: DDGSException: No results found.
[pdf] backend 'startpage' falló: DDGSException: No results found.
[pdf] backend 'mojeek' falló: DDGSException: No results found.
[pdf] backend 'yahoo' falló: DDGSException: No results found.
[pdf] ronda de intento 2 agotó todos los backends — reintentando en 4.1s


KeyboardInterrupt: 

### Descargar los datasets enriquecidos

Los 9 CSV (3 versiones × 3 fracciones) quedaron en `datasets_generados/`. Esta celda los comprime y los descarga a tu computadora.

In [ ]:
import shutil

shutil.make_archive("datasets_generados", "zip", CARPETA_DATASETS)
files.download("datasets_generados.zip")

## 9. Gráficos finales (10% y 50%)

In [ ]:
def graficar_fraccion(resultados, frac):
    subset = resultados[resultados["fraccion"] == frac]
    n_filas = subset["n_filas"].iloc[0]

    colores = {"pandas": "#2563eb", "arrays": "#16a34a", "for_plano": "#dc2626"}
    fig, ax = plt.subplots(figsize=(6, 4.5))
    barras = ax.bar(
        subset["version"],
        subset["segundos"],
        color=[colores[v] for v in subset["version"]],
    )
    ax.bar_label(barras, fmt="%.2fs", padding=3)
    ax.set_title(f"Tiempo de ejecución al {int(frac*100)}% del dataset (n={n_filas})")
    ax.set_ylabel("Segundos")
    ax.set_xlabel("Versión")
    fig.tight_layout()
    fig.savefig(f"grafico_{int(frac*100)}pct.png", dpi=150)
    plt.show()

graficar_fraccion(resultados, 0.01)

In [ ]:
graficar_fraccion(resultados, 0.1)

## 10. Conclusión

Como el cuello de botella es la **latencia de red** de cada búsqueda en `ddgs` (I/O bound), las 3 versiones deberían dar tiempos muy similares: `apply`, `map` y `for` recorren exactamente la misma cantidad de filas haciendo exactamente una request secuencial cada una. La diferencia entre pandas/arrays/for es sobre todo de **estilo de código**, no de performance real acá.

Si el objetivo fuera realmente acelerar esto, el camino es paralelizar las búsquedas con `concurrent.futures.ThreadPoolExecutor`, no cambiar la forma de iterar.

In [27]:
%rm -rf datasets_generados/